<a href="https://colab.research.google.com/github/EvenSol/NeqSim-Colab/blob/master/notebooks/risk/operational_risk_and_availability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Operational risk, availability, and production impact with NeqSim

This notebook demonstrates how the NeqSim risk framework can be coupled directly to a process model. We build a small gas-processing train, quantify the production effect of equipment failures, use reliability data, and run a Monte Carlo availability study.

The workflow is intended for teaching, concept screening, RAM studies, brownfield debottlenecking, and agentic engineering demonstrations. Reliability inputs used below are illustrative unless explicitly replaced by project-approved data.

Documentation: https://equinor.github.io/neqsim/risk/index.html

In [ ]:
%pip -q install neqsim

from neqsim import jneqsim
import json
import pandas as pd
import matplotlib.pyplot as plt

SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
ProcessSystem = jneqsim.process.processmodel.ProcessSystem
Stream = jneqsim.process.equipment.stream.Stream
Separator = jneqsim.process.equipment.separator.Separator
Compressor = jneqsim.process.equipment.compressor.Compressor
Cooler = jneqsim.process.equipment.heatexchanger.Cooler
ProductionImpactAnalyzer = jneqsim.process.util.optimizer.ProductionImpactAnalyzer
DegradedOperationOptimizer = jneqsim.process.util.optimizer.DegradedOperationOptimizer
OperationalRiskSimulator = jneqsim.process.safety.risk.OperationalRiskSimulator
ReliabilityDataSource = jneqsim.process.equipment.failure.ReliabilityDataSource

print('NeqSim operational risk classes loaded')

## 1. Build the physical process model

The risk calculation starts from a normal NeqSim `ProcessSystem`. This is important: equipment criticality is evaluated against the same process model used for thermodynamics and process simulation.

In [ ]:
fluid = SystemSrkEos(308.15, 55.0)
fluid.addComponent('methane', 0.86)
fluid.addComponent('ethane', 0.08)
fluid.addComponent('propane', 0.035)
fluid.addComponent('n-butane', 0.015)
fluid.addComponent('CO2', 0.01)
fluid.setMixingRule('classic')

process = ProcessSystem()
feed = Stream('Well Feed', fluid)
feed.setFlowRate(100000.0, 'kg/hr')
feed.setTemperature(35.0, 'C')
feed.setPressure(55.0, 'bara')
process.add(feed)

separator = Separator('Inlet Separator', feed)
process.add(separator)

compressor = Compressor('Export Compressor', separator.getGasOutStream())
compressor.setOutletPressure(120.0, 'bara')
process.add(compressor)

cooler = Cooler('Export Cooler', compressor.getOutletStream())
cooler.setOutTemperature(35.0, 'C')
process.add(cooler)

export = Stream('Export Gas', cooler.getOutletStream())
process.add(export)
process.run()

baseline = export.getFlowRate('kg/hr')
print(f'Baseline export: {baseline:,.0f} kg/h')
print(f'Compressor power: {compressor.getPower("kW"):,.0f} kW')
assert baseline > 0.0

## 2. Production-impact and equipment-criticality analysis

`ProductionImpactAnalyzer` perturbs the process topology to estimate how failure of a named unit affects the selected product stream. This gives a physics-coupled production consequence rather than assigning every failure the same consequence.

In [ ]:
analyzer = ProductionImpactAnalyzer(process)
analyzer.setFeedStreamName('Well Feed')
analyzer.setProductStreamName('Export Gas')

impact = analyzer.analyzeFailureImpact('Export Compressor')
print('Export compressor failure')
print(f'  baseline production: {impact.getBaselineProduction():,.0f} kg/h')
print(f'  failed production:   {impact.getProductionWithFailure():,.0f} kg/h')
print(f'  production loss:     {impact.getProductionLossPercent():.1f}%')
print(f'  recommended action:  {impact.getRecommendedAction()}')

criticality = dict(analyzer.rankEquipmentByCriticality())
criticality_df = (pd.DataFrame(criticality.items(), columns=['equipment', 'production_loss_percent'])
                    .sort_values('production_loss_percent', ascending=False))
display(criticality_df)

criticality_df.plot.bar(x='equipment', y='production_loss_percent', legend=False, figsize=(8,4))
plt.ylabel('Production loss [%]')
plt.title('Process-coupled equipment criticality')
plt.tight_layout()
plt.show()

## 3. Degraded-operation study

A failure does not always require a full shutdown. The degraded-operation optimizer can be used to search for a feasible reduced-capacity state and to produce a recovery-plan object.

In [ ]:
optimizer = DegradedOperationOptimizer(process)
optimizer.setFeedStreamName('Well Feed')
optimizer.setProductStreamName('Export Gas')

degraded = optimizer.optimizeWithEquipmentDown('Export Compressor')
print(f'Capacity factor: {100.0 * degraded.getCapacityFactor():.1f}%')
print(f'Optimal feed rate: {degraded.getOptimalFlowRate():,.0f} kg/h')
print(f'Optimal production: {degraded.getOptimalProduction():,.0f} kg/h')
print('\nRecovery plan:')
print(optimizer.createRecoveryPlan('Export Compressor').toString())

## 4. Reliability data and Monte Carlo production availability

NeqSim provides a reliability-data source and an `OperationalRiskSimulator`. For real project work, replace illustrative/default reliability values with approved project or company data and document the provenance.

In [ ]:
reliability = ReliabilityDataSource.getInstance()
comp_rel = reliability.getReliabilityData('Compressor', 'Centrifugal')
print('Example reliability record for centrifugal compressor')
print(f'  MTBF: {comp_rel.getMtbf():,.0f} h')
print(f'  MTTR: {comp_rel.getMttr():,.0f} h')
print(f'  availability: {100.0 * comp_rel.getAvailability():.2f}%')

risk_sim = OperationalRiskSimulator(process)
risk_sim.setFeedStreamName('Well Feed')
risk_sim.setProductStreamName('Export Gas')
risk_sim.setRandomSeed(42)
risk_sim.addEquipmentMtbf('Export Compressor', 25000.0, 72.0)
risk_sim.addEquipmentMtbf('Export Cooler', 50000.0, 24.0)
risk_sim.addEquipmentMtbf('Inlet Separator', 250000.0, 48.0)

mc = risk_sim.runSimulation(1000, 365)
print('\nMonte Carlo result')
print(f'  mean availability: {mc.getMeanAvailability():.2f}%')
print(f'  production efficiency: {mc.getProductionEfficiency():.2f}%')
print(f'  mean failures/year: {mc.getMeanFailureCount():.2f}')
print(f'  mean downtime: {mc.getMeanDowntimeHours():.1f} h')
print(f'  production P10/P50/P90: {mc.getP10Production():,.0f} / {mc.getP50Production():,.0f} / {mc.getP90Production():,.0f}')

report = json.loads(mc.toJson())
print('\nAvailable JSON result fields:', sorted(report.keys()))
assert 0.0 <= mc.getMeanAvailability() <= 100.0

## 5. How this fits an agentic NeqSim workflow

A useful agentic pattern is: **detect changed condition → update process model → evaluate equipment consequence → run reliability/risk scenario → propose degraded operating point → return auditable result object**. The process model remains the physics authority while the agent coordinates scenarios and decisions.

### Suggested extensions
- compare 1×100% and 2×50% compressor configurations;
- propagate uncertain MTBF/MTTR assumptions;
- convert production loss to economic consequence;
- connect condition monitoring to remaining-useful-life models;
- use the dynamic risk framework where startup and shutdown transients materially affect loss.

This notebook is a demonstration of software capability, not a replacement for a project RAM study or approved reliability database.